# 02 - Merge, Filter and Build Analysis Dataset
Project: MetS Risk Score - Multimodel prediction in young adults
Input: 15 raw NHANES P_ CSVs from data/raw
Output: Single analysis ready dataset

In [1]:
# Cell 2: Mount Drive and Define Paths
from google.colab import drive
drive.mount('/content/drive')

import os

Drive_root = '/content/drive/MyDrive/mets-risk-score/'
data_raw = os.path.join(Drive_root, 'data/raw/')
data_proc = os.path.join(Drive_root, 'data/processed/')
fig_dir = os.path.join(Drive_root, 'outputs/figures/')

print("Drive mounted sucessfully")
print(f"Raw Data: {data_raw}")
print(f"Processed: {data_proc}")

Mounted at /content/drive
Drive mounted sucessfully
Raw Data: /content/drive/MyDrive/mets-risk-score/data/raw/
Processed: /content/drive/MyDrive/mets-risk-score/data/processed/


In [2]:
# Cell 3: Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)


In [3]:
# Cell 4: Load all 15 tables from drive

table_names = [
    'DEMO',    # Demographics
    'BIOPRO',  # Secondary biochemistry
    'CBC',     # Complete blood count
    'DR1TOT',  # Dietary recall day 1
    'DR2TOT',  # Dietary recall day 2
    'PAQ',     # Physical activity
    'SLQ',     # Sleep
    'DPQ',     # Mental health (PHQ-9)
    'SMQ',     # Smoking
    'ALQ',     # Alcohol
    'BMX',     # Body measures
    'TRIGLY',  # Triglycerides (target component)
    'HDL',     # HDL cholesterol (target component)
    'GLU',     # Fasting glucose (target component)
    'BPX',     # Blood pressure (target component)
]

# For clean and aligned columns, looks like a proper table
print(f"{'Table':<10} {'Rows':>8} {'Columns':>10}")

tables = {}

for name in table_names:
  path = os.path.join(data_raw, f"{name}.csv")

  if not os.path.exists(path):
    print(f"{name:<10} file not found at {path}")
    continue

  df = pd.read_csv(path, low_memory= False)

  # Ensure SEQN is an integer for clean merging
  df['SEQN'] = df['SEQN'].astype(int)

  tables[name] = df
  print(f"{name:<10} {len(df):>8,} {len(df.columns):>10}")

print(f"\n {len(tables)} tables loaded into memory")

Table          Rows    Columns
DEMO         15,560         29
BIOPRO       10,409         41
CBC          13,772         22
DR1TOT       14,300        168
DR2TOT       14,300         85
PAQ           9,693         17
SLQ          10,195         11
DPQ           8,965         11
SMQ          11,137         16
ALQ           8,965         10
BMX          14,300         22
TRIGLY        5,090         10
HDL          12,198          3
GLU           5,090          4
BPX          11,656         12

 15 tables loaded into memory


In [4]:
# Cell 5: Merge all tables on SEQN

# Start with demographics as the base
merged = tables['DEMO'].copy()
print(f"Base  (DEMO): {len(merged):>6,} participats")

# left join each additional table
for name in table_names:
  if name == 'DEMO':
    continue

  df = tables[name]

  # Identify columns that would duplicate (other than SEQN)
  # Add suffix to avoid column name collisions across tables
  before_cols = len(merged.columns)
  merged = merged.merge(df, on = 'SEQN', how = 'left', suffixes = ('', f'_{name}'))
  added = len(merged.columns) - before_cols

  print(f"   + {name:<10}     : {len(merged):>6,} rows | +{added} columns")

Base  (DEMO): 15,560 participats
   + BIOPRO         : 15,560 rows | +40 columns
   + CBC            : 15,560 rows | +21 columns
   + DR1TOT         : 15,560 rows | +167 columns
   + DR2TOT         : 15,560 rows | +84 columns
   + PAQ            : 15,560 rows | +16 columns
   + SLQ            : 15,560 rows | +10 columns
   + DPQ            : 15,560 rows | +10 columns
   + SMQ            : 15,560 rows | +15 columns
   + ALQ            : 15,560 rows | +9 columns
   + BMX            : 15,560 rows | +21 columns
   + TRIGLY         : 15,560 rows | +9 columns
   + HDL            : 15,560 rows | +2 columns
   + GLU            : 15,560 rows | +3 columns
   + BPX            : 15,560 rows | +11 columns


In [5]:
# Cell 6: Filter to Young Adults Age 18-45
# RIDAGEYR = age in years at screening (From DEMO table)

before = len(merged)
age_filtered = merged[(merged['RIDAGEYR'] >=18) & (merged['RIDAGEYR'] <=45)].copy()
after = len(age_filtered)
removed = before - after

print(f"   Before filter : {before:>6,} participants")
print(f"   After filter  : {after:>6,} participants")
print(f"   Removed       : {removed:>6,} (outside 18–45 age range)")

   Before filter : 15,560 participants
   After filter  :  4,142 participants
   Removed       : 11,418 (outside 18–45 age range)


In [7]:
# Cell 7: Apply Exclusion Criteria

# We exclude participants who would make the analysis unreliable or biased.
# Each exclusion is documented with a reason — this becomes your
# "Study Population" paragraph in the methods section.
#
# RIDSTATR = interview/exam status
#   1 = interviewed only (no physical exam — missing lab values)
#   2 = interviewed AND examined (complete data) ← keep these only
#
# RIAGENDR = gender (1=Male, 2=Female)
# RIDEXPRG = pregnancy status (1=pregnant) ← exclude pregnant participants
#            because pregnancy significantly alters metabolic markers

cohort = age_filtered.copy()
exclusion_log =[]

def exclude(df, condition, reason):
  """Apply Exclusion, log how many were removed, reture filterd df"""
  before = len(df)
  df_filtered = df[~condition].copy()
  after = len(df_filtered)
  removed = before - after
  exclusion_log.append({
      'Reason': reason,
      'Removed': removed,
      'Remaining': after
  })
  return df_filtered

# Exclusion 1: Exam-only participants (no lab data)
cohort = exclude(
    cohort,
    cohort['RIDSTATR'] != 2,
    'No physical examination(interview only)'
)

# Exclusion 2: Pregnenet Participants
# RIDEXPRG = 1 means confirmed pregnent at exam
cohort = exclude(
    cohort,
    cohort['RIDEXPRG'] == 1,
    'Pregnent at the time of examination'
)

# Print Exclusive table
print(f"{'Step':<5} {'Reason':<45} {'Removed':>8} {'Remaining':>10}")
print("─" * 72)

start = len(age_filtered)
print(f"{'0':<5} {'Age-filtered cohort (18-45)':<45} {'—':>8} {start:>10,}")

for i, row in enumerate(exclusion_log, 1):
    print(f"{i:<5} {row['Reason']:<45} {row['Removed']:>8,} {row['Remaining']:>10,}")

print("─" * 72)
print(f"\n Final cohort after exclusions: {len(cohort):,} participants")


Step  Reason                                         Removed  Remaining
────────────────────────────────────────────────────────────────────────
0     Age-filtered cohort (18-45)                          —      4,142
1     No physical examination(interview only)            305      3,837
2     Pregnent at the time of examination                 87      3,750
────────────────────────────────────────────────────────────────────────

 Final cohort after exclusions: 3,750 participants


In [9]:
# Cell 8: Cohort Summary
# Quick demographic screen shot of who's in the final cohort

print(f"Total Participants: {len(cohort):,}")
print(f"Total Variables: {cohort.shape[1]:,}")

# Sex distribution
# RIAGENDR: 1 = Male, 2 = Female
sex_counts = cohort['RIAGENDR'].value_counts()
print(f"Male   : {sex_counts.get(1.0, 0):,} ({sex_counts.get(1.0, 0)/len(cohort)*100:.1f}%)")
print(f"Female : {sex_counts.get(2.0, 0):,} ({sex_counts.get(2.0, 0)/len(cohort)*100:.1f}%)")

# Race/Ethnicity
# RIDRETH3: 1=Mexican American, 2=Other Hispanic, 3=Non-Hispanic White,
#           4=Non-Hispanic Black, 6=Non-Hispanic Asian, 7=Other/Multi
race_map = {
    1.0: 'Mexican American',
    2.0: 'Other Hispanic',
    3.0: 'Non-Hispanic White',
    4.0: 'Non-Hispanic Black',
    6.0: 'Non-Hispanic Asian',
    7.0: 'Other/Multiracial'
}
print(f"\n   Race/Ethnicity:")
for code, label in race_map.items():
    n = (cohort['RIDRETH3'] == code).sum()
    pct = n / len(cohort) * 100
    print(f"   {label:<25} : {n:,} ({pct:.1f}%)")

# Age
print(f" Mean ± SD : {cohort['RIDAGEYR'].mean():.1f} ± {cohort['RIDAGEYR'].std():.1f}")
print(f" Median    : {cohort['RIDAGEYR'].median():.1f}")

# Missingness overview
total_cells = cohort.shape[0] * cohort.shape[1]
missing_cells = cohort.isnull().sum().sum()
print(f"\n  Overall missingness : {missing_cells/total_cells*100:.1f}% of all values")

Total Participants: 3,750
Total Variables: 447
Male   : 1,812 (48.3%)
Female : 1,938 (51.7%)

   Race/Ethnicity:
   Mexican American          : 551 (14.7%)
   Other Hispanic            : 404 (10.8%)
   Non-Hispanic White        : 1,119 (29.8%)
   Non-Hispanic Black        : 959 (25.6%)
   Non-Hispanic Asian        : 488 (13.0%)
   Other/Multiracial         : 229 (6.1%)
 Mean ± SD : 31.2 ± 8.4
 Median    : 31.0

  Overall missingness : 30.0% of all values
